In [1]:
from google.colab import files
import os

print("Please upload your 'model.pkl' file:")
uploaded = files.upload()

for fn in uploaded.keys():
  print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))

# Rename the uploaded file to 'model.pkl' if it has a different name
if 'model.pkl' not in os.listdir('.') and len(uploaded) > 0:
    uploaded_file = list(uploaded.keys())[0]
    os.rename(uploaded_file, 'model.pkl')
    print(f"Renamed {uploaded_file} to model.pkl")

Please upload your 'model.pkl' file:


Saving iris_model.pkl to iris_model (1).pkl
User uploaded file "iris_model (1).pkl" with length 151815 bytes
Renamed iris_model (1).pkl to model.pkl


In [2]:
import joblib
import pandas as pd
import os

# Standard loading procedure without any Azure mocks
model_file = 'model.pkl'

if os.path.exists(model_file):
    try:
        print(f"Attempting to load {model_file} using joblib...")
        model = joblib.load(model_file)
        print("✅ Success! The model was loaded directly.")

        # Display the type of the model object to see what it actually is
        print(f"Model type: {type(model)}")

    except Exception as e:
        print(f"❌ Direct load failed: {e}")
        print("\nThis usually happens because the pickle file contains custom class definitions ")
        print("(like Azure-specific wrappers) that aren't available in the current environment.")
else:
    print(f"File {model_file} not found in /content/")

Attempting to load model.pkl using joblib...
✅ Success! The model was loaded directly.
Model type: <class 'sklearn.ensemble._forest.RandomForestClassifier'>


/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.8.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.8.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


### Test the Model with Sample Data
We will create a small DataFrame with the expected feature names for an Iris classification task and use the loaded model to predict the class.

In [3]:
import pandas as pd
import numpy as np

# Define sample data matching the Iris dataset structure
data = {
    'sepal length (cm)': [5.1, 5.9, 6.9],
    'sepal width (cm)': [3.5, 3.0, 3.1],
    'petal length (cm)': [1.4, 4.2, 5.4],
    'petal width (cm)': [0.2, 1.5, 2.1]
}

# Convert to DataFrame
test_df = pd.DataFrame(data)

try:
    # Make predictions
    predictions = model.predict(test_df)

    # Get probabilities if the model supports it
    if hasattr(model, 'predict_proba'):
        probabilities = model.predict_proba(test_df)
        print("Probabilities per class:")
        print(probabilities)

    # Display results
    test_df['predicted_class'] = predictions
    print("\nTest Results:")
    display(test_df)

except Exception as e:
    print(f"❌ Prediction failed: {e}")
    print("Note: Ensure the feature names in 'data' match exactly what the model was trained with.")

Probabilities per class:
[[1.   0.   0.  ]
 [0.   0.99 0.01]
 [0.   0.01 0.99]]

Test Results:


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),predicted_class
0,5.1,3.5,1.4,0.2,0
1,5.9,3.0,4.2,1.5,1
2,6.9,3.1,5.4,2.1,2


### Extended Model Testing
Generating a larger batch of synthetic data to observe model behavior across a wider variety of feature combinations.

In [4]:
import pandas as pd
import numpy as np

# Generate 15 sample records with random variations based on typical Iris ranges
np.random.seed(42)
n_samples = 15

extended_data = {
    'sepal length (cm)': np.random.uniform(4.3, 7.9, n_samples),
    'sepal width (cm)': np.random.uniform(2.0, 4.4, n_samples),
    'petal length (cm)': np.random.uniform(1.0, 6.9, n_samples),
    'petal width (cm)': np.random.uniform(0.1, 2.5, n_samples)
}

extended_df = pd.DataFrame(extended_data)

try:
    # Perform bulk prediction
    preds = model.predict(extended_df)
    extended_df['predicted_class'] = preds

    print(f"Successfully processed {n_samples} samples.")
    display(extended_df)

    # Show class distribution
    print("\nPredicted Class Counts:")
    print(extended_df['predicted_class'].value_counts())

except Exception as e:
    print(f"Bulk prediction failed: {e}")

Successfully processed 15 samples.


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),predicted_class
0,5.648344,2.440171,4.584515,1.690053,1
1,7.722572,2.730181,2.006092,0.848107,1
2,6.935178,3.259415,1.383804,1.348163,1
3,6.455171,3.036668,6.598425,1.412105,2
4,4.861667,2.698950,6.697229,0.543651,0
5,4.861580,3.468447,5.769544,2.427003,2
6,4.509101,2.334785,2.797221,1.960319,2
7,7.418234,2.701147,1.576265,2.354797,0
8,6.464014,2.879268,5.036975,2.247586,2
9,6.849061,3.094568,3.596900,1.534960,1



Predicted Class Counts:
predicted_class
1    6
0    5
2    4
Name: count, dtype: int64
